# Day 2: Mechanical Material Selection & Performance Ranking Engine
**Author:** Navneet Namdeo  
**Domain:** Material Science / Structural Optimization  
**Repository:** Machine-Learning-Projects

### Overview
This notebook builds a Pandas-based engineering data pipeline to store, analyze, and rank structural engineering materials (steels, aluminum alloys, titanium, composites). It computes material performance indices (Specific Stiffness and Specific Strength) to drive lightweight design decisions.

In [2]:
import pandas as pd
import numpy as np

# 1. Construct a comprehensive engineering material database
material_data = {
    "Material_Name": [
        "Structural Steel (A36)",
        "High-Strength Steel (AISI 4340)",
        "Aluminum 6061-T6",
        "Aluminum 7075-T6",
        "Titanium Ti-6Al-4V (Grade 5)",
        "Carbon Fiber Composite (CFRP)",
        "Inconel 718",
        "Magnesium AZ31B-H24"
    ],
    "Material_Class": [
        "Steel", "Steel", "Aluminum", "Aluminum", 
        "Titanium", "Composite", "Nickel Superalloy", "Magnesium"
    ],
    "Density_kg_m3": [7850, 7850, 2700, 2810, 4430, 1600, 8190, 1770],
    "Youngs_Modulus_GPa": [200, 210, 68.9, 71.7, 113.8, 150, 205, 45],
    "Yield_Strength_MPa": [250, 860, 276, 503, 880, 900, 1100, 220],
    "UTS_MPa": [400, 1280, 310, 572, 950, 1200, 1375, 290],
    "Poissons_Ratio": [0.26, 0.29, 0.33, 0.33, 0.34, 0.28, 0.29, 0.35]
}

# 2. Convert to Pandas DataFrame
df = pd.DataFrame(material_data)

# 3. Compute Structural Performance Indices
# Specific Stiffness (E / Density)
df["Specific_Stiffness_GPa_per_g_cm3"] = (df["Youngs_Modulus_GPa"] * 1000) / df["Density_kg_m3"]

# Specific Strength (Yield Strength / Density)
df["Specific_Strength_MPa_per_g_cm3"] = (df["Yield_Strength_MPa"] * 1000) / df["Density_kg_m3"]

# Yield-to-Tensile Ratio
df["Yield_to_UTS_Ratio"] = df["Yield_Strength_MPa"] / df["UTS_MPa"]

# 4. Function to filter and rank materials based on engineering constraints
def rank_materials_for_design(data, min_yield_mpa=0, max_density=10000, rank_by="Specific_Strength_MPa_per_g_cm3"):
    """
    Filters materials by yield strength and density constraints, then ranks them
    by a selected structural performance index.
    """
    filtered_df = data[
        (data["Yield_Strength_MPa"] >= min_yield_mpa) & 
        (data["Density_kg_m3"] <= max_density)
    ].copy()
    
    # Sort in descending order based on chosen performance metric
    ranked_df = filtered_df.sort_values(by=rank_by, ascending=False).reset_index(drop=True)
    
    return ranked_df

# --- DEMONSTRATION 1: Aerospace Lightweight High-Strength Ranking ---
print("=" * 80)
print("     AEROSPACE LIGHTWEIGHT DESIGN RANKING (Ranked by Specific Strength)")
print("=" * 80)
aerospace_ranking = rank_materials_for_design(
    df, min_yield_mpa=250, max_density=5000, rank_by="Specific_Strength_MPa_per_g_cm3"
)
print(aerospace_ranking[["Material_Name", "Material_Class", "Density_kg_m3", "Yield_Strength_MPa", "Specific_Strength_MPa_per_g_cm3"]])

# --- DEMONSTRATION 2: High-Stiffness Structural Beam Ranking ---
print("\n" + "=" * 80)
print("     HIGH-STIFFNESS STRUCTURAL BEAM RANKING (Ranked by Specific Stiffness)")
print("=" * 80)
stiffness_ranking = rank_materials_for_design(
    df, min_yield_mpa=0, max_density=10000, rank_by="Specific_Stiffness_GPa_per_g_cm3"
)
print(stiffness_ranking[["Material_Name", "Material_Class", "Youngs_Modulus_GPa", "Specific_Stiffness_GPa_per_g_cm3"]])

     AEROSPACE LIGHTWEIGHT DESIGN RANKING (Ranked by Specific Strength)
                   Material_Name Material_Class  Density_kg_m3  \
0  Carbon Fiber Composite (CFRP)      Composite           1600   
1   Titanium Ti-6Al-4V (Grade 5)       Titanium           4430   
2               Aluminum 7075-T6       Aluminum           2810   
3               Aluminum 6061-T6       Aluminum           2700   

   Yield_Strength_MPa  Specific_Strength_MPa_per_g_cm3  
0                 900                       562.500000  
1                 880                       198.645598  
2                 503                       179.003559  
3                 276                       102.222222  

     HIGH-STIFFNESS STRUCTURAL BEAM RANKING (Ranked by Specific Stiffness)
                     Material_Name     Material_Class  Youngs_Modulus_GPa  \
0    Carbon Fiber Composite (CFRP)          Composite               150.0   
1  High-Strength Steel (AISI 4340)              Steel               210.0   
2    